# Phase 1 : Modèles Baselines (LR & RF) 🚀

Objectif : 
1. Préparer les données (Preprocessing + SMOTE)
2. Entraîner une Régression Logistique (baseline linéaire)
3. Entraîner un Random Forest (baseline arbre)
4. Évaluer avec les métriques adaptées à la fraude (AUC-PR, F1-Score, Recall)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from loguru import logger
from src.data.loader import DataLoader
from src.data.preprocessor import FraudPreprocessor
from src.models.trainer import ModelTrainer
from src.models.evaluator import ModelEvaluator

# Desactiver les warnings sklearn liés aux features names
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement et Nettoyage des Features

In [ ]:
loader = DataLoader()
df = loader.load()

# On retire les colonnes non prédictives (Identifiants) ou redondantes avec 'hour'/'day_of_week'
cols_to_drop = ['organization', 'transaction_id', 'user_id', 'transaction_timestamp']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

logger.info(f"Features utiles : {len(df_clean.columns) - 1}")
df_clean.head(3)

## 2. Splits : Train / Val / Test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = loader.get_splits(df_clean)

## 3. Preprocessing & SMOTE
**Règle d'or** : SMOTE est appliqué UNIQUEMENT sur le Train Set !

In [ ]:
# Initialisation du preprocessor avec le ratio SMOTE (20%)
preprocessor = FraudPreprocessor(smote_sampling_strategy=0.2)

# Fit + Transform sur Train (génère les nouvelles données minoritaires)
X_train_proc, y_train_proc = preprocessor.fit_transform_train(X_train, y_train)

# Transform (Seulement) sur Val et Test
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

# Sauvegarde du preprocessor pour la Phase 3 (API)
preprocessor.save()

## 4. Modèle Baseline 1 : Régression Logistique
Un modèle linéaire simple, rapide, mais souvent limité pour les patterns complexes de fraude.

In [ ]:
# L'evaluator va calculer les métriques et tracer les courbes
evaluator = ModelEvaluator(threshold=0.5)

logger.info("Entraînement de la Régression Logistique...")
lr_trainer = ModelTrainer(model_name="logistic_regression")
lr_trainer.fit(X_train_proc, y_train_proc)

# Prédictions sur le set de Validation (Pour comparer les modèles entre eux)
y_val_prob_lr = lr_trainer.predict_proba(X_val_proc)[:, 1]

# Evaluation
metrics_lr = evaluator.evaluate(y_val, y_val_prob_lr, model_name="Logistic Regression")
evaluator.plot_precision_recall_curve(y_val, y_val_prob_lr, model_name="Logistic Regression")

## 5. Modèle Baseline 2 : Random Forest
Contrairement à la régression logistique, ce modèle capture les relations non-linéaires.

In [ ]:
logger.info("Entraînement du Random Forest...")
rf_trainer = ModelTrainer(model_name="random_forest")
rf_trainer.fit(X_train_proc, y_train_proc)

y_val_prob_rf = rf_trainer.predict_proba(X_val_proc)[:, 1]

# Evaluation
metrics_rf = evaluator.evaluate(y_val, y_val_prob_rf, model_name="Random Forest")
evaluator.plot_precision_recall_curve(y_val, y_val_prob_rf, model_name="Random Forest")

## 6. Comparatif des Baselines

In [ ]:
results = {
    "Logistic Regression": metrics_lr,
    "Random Forest": metrics_rf
}
evaluator.compare_models(results)

# Sauvegarde des modèles (optionnel à ce stade si on préfère XGBoost par la suite)
# lr_trainer.save()
# rf_trainer.save()